# Actividad 2: Construcción de Muestra Representativa con PySpark
## Dataset: CIC-IDS2017 — Canadian Institute for Cybersecurity

**TC4034.10 Análisis de Grandes Volúmenes de Datos | Grupo 10 | Equipo #5**

- Luz Copelia Minutti Pérez — A01796921
- Karen Cecilia Varela Castro — A00958670
- Kevin Rosario Cota Rodríguez — A01796705
- Carlos Aaron Bocanegra Buitrón — A01796345

**Fecha:** 07 de Mayo de 2026

---

## Descripción general

En este cuaderno implementamos el proceso de particionamiento y muestreo representativo sobre el dataset **CIC-IDS2017**, desarrollado por el Canadian Institute for Cybersecurity de la University of New Brunswick. El dataset contiene flujos de tráfico de red capturados durante cinco días hábiles (lunes a viernes), con tráfico benigno y 14 tipos de ataques cibernéticos etiquetados. Cada registro representa un flujo de red descrito mediante 78 características estadísticas extraídas con la herramienta CICFlowMeter, más la columna `Label`.

**Objetivos del cuaderno:**
1. Cargar y limpiar el dataset en PySpark
2. Derivar las variables de caracterización (`attack_group`, `day_name`)
3. Aplicar las reglas de particionamiento R1–R13 con demostración detallada de cada partición
4. Extraer sub-muestras de verificación por partición

**Enlace al dataset en Google Drive:** https://drive.google.com/drive/folders/1ZaAmUQqSZUGObF3YVubLHyK0QNg4eEMB?usp=drive_link

**Enlace al Jupyter Notebook:** [REEMPLAZAR CON EL LINK DIRECTO AL .ipynb EN DRIVE]

---
## Sección 1: Instalación de PySpark

Dado que ejecutamos este cuaderno en **Google Colab**, el entorno no incluye PySpark de manera predeterminada. Por ello, comenzamos instalándolo con `pip`. La bandera `-q` suprime la salida detallada de la instalación para mantener el cuaderno limpio. Esta celda debe ejecutarse al inicio de cada sesión de Colab, ya que el entorno se reinicia cada vez que se cierra la sesión.

In [1]:
!pip install pyspark -q

---
## Sección 2: Montaje de Google Drive

Para acceder a los archivos CSV del dataset almacenados en nuestra cuenta de Google Drive, montamos la unidad dentro del entorno de Colab. Esto nos permite leer los datos directamente desde Drive sin necesidad de descargarlos al entorno local de Colab, lo que simplifica el flujo de trabajo y evita la pérdida de archivos al reiniciar la sesión.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## Sección 3: Inicialización de la Sesión de Spark

Iniciamos la sesión de Apache Spark mediante `SparkSession`, que es el punto de entrada para todas las operaciones de PySpark. Configuramos 4 GB de memoria para el driver (`spark.driver.memory = 4g`) con el fin de manejar el volumen del dataset (~840 MB en disco). También importamos el módulo `functions as F` con las funciones de transformación y `DoubleType` para el manejo de tipos numéricos.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName('CICIDS2017_Muestreo') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print(f'Spark versión: {spark.version}')
print('Sesión iniciada correctamente.')

Spark versión: 4.0.2
Sesión iniciada correctamente.


---
## Sección 4: Carga del Dataset

Cargamos los 8 archivos CSV del dataset CIC-IDS2017 de manera simultánea apuntando a la carpeta completa con el comodín `*.csv`. Usamos `header=true` para que PySpark reconozca la primera fila como encabezado, e `inferSchema=true` para inferir automáticamente el tipo de dato de cada columna. Aplicamos `strip()` a los nombres de columnas para eliminar los espacios en blanco que el dataset original incluye en sus encabezados CSV.

In [4]:
DATA_PATH = '/content/drive/MyDrive/BigData_PySpark/MachineLearningCVE/*.csv'

df_raw = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(DATA_PATH)

# Quitar espacios en nombres de columnas
df_raw = df_raw.toDF(*[c.strip() for c in df_raw.columns])

print(f'Registros cargados: {df_raw.count():,}')
print(f'Columnas:           {len(df_raw.columns)}')

Registros cargados: 2,830,743
Columnas:           79


---
## Sección 5: Verificación Inicial de Etiquetas

Antes de realizar cualquier transformación, verificamos la distribución de la columna `Label` para confirmar que el dataset se cargó correctamente, entender la composición de la población objetivo e identificar el desbalanceo de clases y los caracteres corruptos en las etiquetas `Web Attack`.

In [5]:
print('Distribución de etiquetas (pre-limpieza)')
df_raw.groupBy('Label') \
      .count() \
      .orderBy('count', ascending=False) \
      .show(20, truncate=False)

Distribución de etiquetas (pre-limpieza)
+--------------------------+-------+
|Label                     |count  |
+--------------------------+-------+
|BENIGN                    |2273097|
|DoS Hulk                  |231073 |
|PortScan                  |158930 |
|DDoS                      |128027 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7938   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1966   |
|Web Attack � Brute Force  |1507   |
|Web Attack � XSS          |652    |
|Infiltration              |36     |
|Web Attack � Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



---
## Sección 6: Limpieza de Datos

Aplicamos cuatro operaciones de limpieza identificadas durante el EDA de la Actividad 1:

1. **Corrección de caracteres corruptos en `Label`:** `regexp_replace` con `[^\x20-\x7E]` elimina cualquier carácter fuera del rango ASCII imprimible.
2. **Eliminación de duraciones negativas:** `Flow Duration < 0` es físicamente imposible (error del sensor CICFlowMeter).
3. **Eliminación de nulos en `Flow Bytes/s`:** 1,358 registros con null por división entre duración cero.
4. **Conversión de infinitos en `Flow Packets/s`:** Valores `Inf` producidos por paquetes/0 se convierten a `null`.

In [6]:
# 1. Corregir caracteres corruptos en Label
df = df_raw.withColumn(
    'Label',
    F.regexp_replace(F.col('Label'), '[^\x20-\x7E]', '')
)

# 2. Eliminar duraciones negativas
df = df.filter(F.col('Flow Duration') >= 0)

# 3. Eliminar nulos en Flow Bytes/s
df = df.filter(F.col('Flow Bytes/s').isNotNull())

# 4. Convertir infinitos en Flow Packets/s a null
df = df.withColumn(
    'Flow Packets/s',
    F.when(F.col('Flow Packets/s') == float('inf'), None)
     .otherwise(F.col('Flow Packets/s').cast(DoubleType()))
)

print(f'Registros después de limpieza: {df.count():,}')
print('Limpieza completada.')

Registros después de limpieza: 2,830,628
Limpieza completada.


---
## Sección 7: Verificación Post-Limpieza

Confirmamos que los caracteres corruptos fueron eliminados de las etiquetas `Web Attack` y que el número de registros es consistente con lo esperado.

In [7]:
print('Distribución de etiquetas (post-limpieza)')
df.groupBy('Label') \
  .count() \
  .orderBy('count', ascending=False) \
  .show(20, truncate=False)

Distribución de etiquetas (post-limpieza)
+-------------------------+-------+
|Label                    |count  |
+-------------------------+-------+
|BENIGN                   |2272982|
|DoS Hulk                 |231073 |
|PortScan                 |158930 |
|DDoS                     |128027 |
|DoS GoldenEye            |10293  |
|FTP-Patator              |7938   |
|SSH-Patator              |5897   |
|DoS slowloris            |5796   |
|DoS Slowhttptest         |5499   |
|Bot                      |1966   |
|Web Attack  Brute Force  |1507   |
|Web Attack  XSS          |652    |
|Infiltration             |36     |
|Web Attack  Sql Injection|21     |
|Heartbleed               |11     |
+-------------------------+-------+



---
## Sección 8: Creación de la Variable `attack_group`

Derivamos la primera variable de caracterización agrupando las 14 etiquetas originales en 9 categorías conceptuales usando `withColumn` + `when`/`otherwise`. Esta agrupación reduce la cardinalidad de 14 a 9 clases, evitando que el particionamiento genere combinaciones con muy pocos registros (FTP-Patator y SSH-Patator → BruteForce; cuatro tipos de DoS → DoS).

In [8]:
df = df.withColumn('attack_group',
    F.when(F.col('Label') == 'BENIGN', 'BENIGN')
     .when(F.col('Label').isin('FTP-Patator','SSH-Patator'), 'BruteForce')
     .when(F.col('Label').isin(
         'DoS Hulk','DoS GoldenEye',
         'DoS slowloris','DoS Slowhttptest'), 'DoS')
     .when(F.col('Label') == 'DDoS', 'DDoS')
     .when(F.col('Label').isin(
         'Web Attack  Brute Force',
         'Web Attack  XSS',
         'Web Attack  Sql Injection'), 'WebAttack')
     .when(F.col('Label') == 'Bot', 'Botnet')
     .when(F.col('Label') == 'PortScan', 'PortScan')
     .when(F.col('Label') == 'Infiltration', 'Infiltration')
     .when(F.col('Label') == 'Heartbleed', 'Heartbleed')
     .otherwise('Other')
)

print('Distribución de attack_group')
df.groupBy('attack_group') \
  .count() \
  .orderBy('count', ascending=False) \
  .show()

Distribución de attack_group
+------------+-------+
|attack_group|  count|
+------------+-------+
|      BENIGN|2272982|
|         DoS| 252661|
|    PortScan| 158930|
|        DDoS| 128027|
|  BruteForce|  13835|
|   WebAttack|   2180|
|      Botnet|   1966|
|Infiltration|     36|
|  Heartbleed|     11|
+------------+-------+



---
## Sección 9: Exploración de Columnas Disponibles

Verificamos los nombres exactos de todas las columnas. Esta exploración fue necesaria porque al intentar crear `protocol_name` a partir de la columna `Protocol`, obtuvimos un `AnalysisException`: dicha columna no existe en esta versión ML del dataset. Ajustamos la estrategia usando únicamente `attack_group` y `day_name`.

In [9]:
print(f'Total de columnas disponibles: {len(df.columns)}')
print('\nListado completo de columnas:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

Total de columnas disponibles: 80

Listado completo de columnas:
   1. Destination Port
   2. Flow Duration
   3. Total Fwd Packets
   4. Total Backward Packets
   5. Total Length of Fwd Packets
   6. Total Length of Bwd Packets
   7. Fwd Packet Length Max
   8. Fwd Packet Length Min
   9. Fwd Packet Length Mean
  10. Fwd Packet Length Std
  11. Bwd Packet Length Max
  12. Bwd Packet Length Min
  13. Bwd Packet Length Mean
  14. Bwd Packet Length Std
  15. Flow Bytes/s
  16. Flow Packets/s
  17. Flow IAT Mean
  18. Flow IAT Std
  19. Flow IAT Max
  20. Flow IAT Min
  21. Fwd IAT Total
  22. Fwd IAT Mean
  23. Fwd IAT Std
  24. Fwd IAT Max
  25. Fwd IAT Min
  26. Bwd IAT Total
  27. Bwd IAT Mean
  28. Bwd IAT Std
  29. Bwd IAT Max
  30. Bwd IAT Min
  31. Fwd PSH Flags
  32. Bwd PSH Flags
  33. Fwd URG Flags
  34. Bwd URG Flags
  35. Fwd Header Length34
  36. Bwd Header Length
  37. Fwd Packets/s
  38. Bwd Packets/s
  39. Min Packet Length
  40. Max Packet Length
  41. Packet Length Mean

---
## Sección 10: Creación de la Variable `day_name`

Derivamos la segunda variable de caracterización usando `input_file_name()` de PySpark, que retorna la ruta del archivo CSV de origen de cada registro. Como los archivos están nombrados con el día de la semana, `contains()` extrae el día correspondiente. Esta variable es fundamental porque el comportamiento del dataset varía por día: el lunes contiene exclusivamente tráfico benigno; los ataques se distribuyen de martes a viernes según el cronograma oficial.

In [10]:
df = df.withColumn('day_name',
    F.when(F.input_file_name().contains('Monday'),    'Monday')
     .when(F.input_file_name().contains('Tuesday'),   'Tuesday')
     .when(F.input_file_name().contains('Wednesday'), 'Wednesday')
     .when(F.input_file_name().contains('Thursday'),  'Thursday')
     .when(F.input_file_name().contains('Friday'),    'Friday')
     .otherwise('Unknown')
)

print('Distribución por día')
df.groupBy('day_name').count().orderBy('count', ascending=False).show()

Distribución por día
+---------+------+
| day_name| count|
+---------+------+
|   Friday|703198|
|Wednesday|692682|
|   Monday|529903|
| Thursday|458953|
|  Tuesday|445892|
+---------+------+



---
## Sección 11: Verificación Cruzada día × tipo de ataque

Verificamos la coherencia lógica del particionamiento cruzando `day_name` con `attack_group`. `attack_group` y `day_name` no son variables independientes: el cronograma de captura determina qué ataques aparecen cada día. El resultado esperado es que el lunes solo tenga BENIGN, y que cada tipo de ataque aparezca únicamente en el día correcto según la documentación oficial del CIC-IDS2017.

In [11]:
print('Verificación cruzada: day_name × attack_group')
df.groupBy('day_name', 'attack_group') \
  .count() \
  .orderBy('day_name', 'count', ascending=False) \
  .show(50, truncate=False)

Verificación cruzada: day_name × attack_group
+---------+------------+------+
|day_name |attack_group|count |
+---------+------------+------+
|Wednesday|BENIGN      |440010|
|Wednesday|DoS         |252661|
|Wednesday|Heartbleed  |11    |
|Tuesday  |BENIGN      |432057|
|Tuesday  |BruteForce  |13835 |
|Thursday |BENIGN      |456737|
|Thursday |WebAttack   |2180  |
|Thursday |Infiltration|36    |
|Monday   |BENIGN      |529903|
|Friday   |BENIGN      |414275|
|Friday   |PortScan    |158930|
|Friday   |DDoS        |128027|
|Friday   |Botnet      |1966  |
+---------+------------+------+



---
## Sección 12: Función Genérica de Particionamiento

Implementamos `get_partition()`, función reutilizable que aplica las reglas de particionamiento mediante filtros sobre las dos variables de caracterización. PySpark aplica estos filtros de forma **lazy** (diferida): el procesamiento real no ocurre hasta invocar una Action como `count()` o `show()`.

### Reglas de particionamiento definidas:
| Regla | attack_group | day_name | Descripción |
|-------|-------------|----------|-------------|
| R1  | BENIGN      | Monday    | Tráfico benigno del lunes |
| R2  | BruteForce  | Tuesday   | FTP-Patator + SSH-Patator |
| R3  | DoS         | Wednesday | 4 tipos de DoS |
| R4  | Heartbleed  | Wednesday | Ultra-minoritaria (n=11) |
| R5  | WebAttack   | Thursday  | XSS + BruteForce HTTP + SQLi |
| R6  | Infiltration| Thursday  | Ultra-minoritaria (n=36) |
| R7  | Botnet      | Friday    | Bot |
| R8  | DDoS        | Friday    | DDoS masivo |
| R9  | PortScan    | Friday    | Escaneo de puertos |
| R10 | BENIGN      | Tuesday   | BENIGN coexistente con BruteForce |
| R11 | BENIGN      | Wednesday | BENIGN coexistente con DoS |
| R12 | BENIGN      | Thursday  | BENIGN coexistente con WebAttack |
| R13 | BENIGN      | Friday    | BENIGN coexistente con DDoS/PortScan/Botnet |

In [12]:
def get_partition(df, attack_grp, day):
    """Filtra el DataFrame por attack_group y day_name para generar una partición."""
    return df.filter(
        (F.col('attack_group') == attack_grp) &
        (F.col('day_name')     == day)
    )

# Generar las 13 particiones
df_R1  = get_partition(df, 'BENIGN',      'Monday')
df_R2  = get_partition(df, 'BruteForce',  'Tuesday')
df_R3  = get_partition(df, 'DoS',         'Wednesday')
df_R4  = get_partition(df, 'Heartbleed',  'Wednesday')
df_R5  = get_partition(df, 'WebAttack',   'Thursday')
df_R6  = get_partition(df, 'Infiltration','Thursday')
df_R7  = get_partition(df, 'Botnet',      'Friday')
df_R8  = get_partition(df, 'DDoS',        'Friday')
df_R9  = get_partition(df, 'PortScan',    'Friday')
df_R10 = get_partition(df, 'BENIGN',      'Tuesday')
df_R11 = get_partition(df, 'BENIGN',      'Wednesday')
df_R12 = get_partition(df, 'BENIGN',      'Thursday')
df_R13 = get_partition(df, 'BENIGN',      'Friday')

print('13 particiones definidas correctamente (evaluación lazy — aún no ejecutadas).')

13 particiones definidas correctamente (evaluación lazy — aún no ejecutadas).


---
## Sección 13: Tabla de Probabilidades Empíricas

Calculamos el número de registros de cada partición y su probabilidad empírica P(R) = N_R / N_D. Las probabilidades se calculan directamente desde los datos (no multiplicando marginales) porque `attack_group` y `day_name` están correlacionadas por diseño. La suma debe ser exactamente **1.0000** para confirmar que el esquema es mutuamente excluyente y colectivamente exhaustivo.

In [13]:
total = df.count()

particiones = {
    'R1  BENIGN/Monday':        df_R1,
    'R2  BruteForce/Tuesday':   df_R2,
    'R3  DoS/Wednesday':        df_R3,
    'R4  Heartbleed/Wednesday': df_R4,
    'R5  WebAttack/Thursday':   df_R5,
    'R6  Infiltration/Thursday':df_R6,
    'R7  Botnet/Friday':        df_R7,
    'R8  DDoS/Friday':          df_R8,
    'R9  PortScan/Friday':      df_R9,
    'R10 BENIGN/Tuesday':       df_R10,
    'R11 BENIGN/Wednesday':     df_R11,
    'R12 BENIGN/Thursday':      df_R12,
    'R13 BENIGN/Friday':        df_R13,
}

print(f'{"Partición":<32} {"N registros":>12} {"P(partición)":>13}')
print('-' * 59)
suma = 0
conteos = {}
for nombre, part in particiones.items():
    n = part.count()
    conteos[nombre] = n
    suma += n
    print(f'{nombre:<32} {n:>12,} {n/total:>13.4f}')
print('-' * 59)
print(f'{"TOTAL":<32} {suma:>12,} {suma/total:>13.4f}')
print(f'\n Suma de probabilidades = {suma/total:.4f} (debe ser 1.0000)')

Partición                         N registros  P(partición)
-----------------------------------------------------------
R1  BENIGN/Monday                     529,903        0.1872
R2  BruteForce/Tuesday                 13,835        0.0049
R3  DoS/Wednesday                     252,661        0.0893
R4  Heartbleed/Wednesday                   11        0.0000
R5  WebAttack/Thursday                  2,180        0.0008
R6  Infiltration/Thursday                  36        0.0000
R7  Botnet/Friday                       1,966        0.0007
R8  DDoS/Friday                       128,027        0.0452
R9  PortScan/Friday                   158,930        0.0561
R10 BENIGN/Tuesday                    432,057        0.1526
R11 BENIGN/Wednesday                  440,010        0.1554
R12 BENIGN/Thursday                   456,737        0.1614
R13 BENIGN/Friday                     414,275        0.1464
-----------------------------------------------------------
TOTAL                               2,83

---
## Sección 14: Demostración Detallada — Particiones BENIGN

Para cada partición de tráfico benigno mostramos: (1) el número de registros, (2) una muestra de 5 filas con las columnas más relevantes, y (3) estadísticas básicas de `Flow Duration` y `Flow Packets/s` para caracterizar el comportamiento del flujo en esa partición.

In [14]:
# Columnas a mostrar en los ejemplos
COLS_DEMO = ['Label', 'attack_group', 'day_name',
             'Flow Duration', 'Total Fwd Packets',
             'Total Backward Packets', 'Flow Packets/s']

benign_parts = [
    ('R1  BENIGN/Monday',    df_R1),
    ('R10 BENIGN/Tuesday',   df_R10),
    ('R11 BENIGN/Wednesday', df_R11),
    ('R12 BENIGN/Thursday',  df_R12),
    ('R13 BENIGN/Friday',    df_R13),
]

for nombre, part in benign_parts:
    n = conteos[nombre]
    print(f'\n' + '='*70)
    print(f'  PARTICIÓN {nombre}')
    print(f'  Registros: {n:,} | P = {n/total:.4f}')
    print('='*70)

    print('\n Distribución de etiquetas (Label)')
    part.groupBy('Label').count() \
        .orderBy('count', ascending=False).show(5, truncate=False)

    print('Muestra de 5 registros')
    part.select(COLS_DEMO).show(5, truncate=True)

    print('Estadísticas de Flow Duration y Flow Packets/s ')
    part.select('Flow Duration', 'Flow Packets/s') \
        .describe().show(truncate=False)


  PARTICIÓN R1  BENIGN/Monday
  Registros: 529,903 | P = 0.1872

 Distribución de etiquetas (Label)
+------+------+
|Label |count |
+------+------+
|BENIGN|529903|
+------+------+

Muestra de 5 registros
+------+------------+--------+-------------+-----------------+----------------------+----------------+
| Label|attack_group|day_name|Flow Duration|Total Fwd Packets|Total Backward Packets|  Flow Packets/s|
+------+------------+--------+-------------+-----------------+----------------------+----------------+
|BENIGN|      BENIGN|  Monday|            4|                2|                     0|        500000.0|
|BENIGN|      BENIGN|  Monday|            1|                2|                     0|       2000000.0|
|BENIGN|      BENIGN|  Monday|            1|                2|                     0|       2000000.0|
|BENIGN|      BENIGN|  Monday|            1|                2|                     0|       2000000.0|
|BENIGN|      BENIGN|  Monday|            3|                2|            

---
## Sección 15: Demostración Detallada — Particiones de ATAQUE

Para cada partición de ataque mostramos: (1) el número de registros, (2) la distribución exacta de sub-clases (`Label`) dentro de la partición, (3) una muestra de 5 filas con columnas relevantes, y (4) estadísticas de `Flow Duration` y `Flow Packets/s` que permiten observar el comportamiento estadístico diferenciado de cada tipo de ataque.

In [15]:
ataque_parts = [
    ('R2  BruteForce/Tuesday',   df_R2),
    ('R3  DoS/Wednesday',        df_R3),
    ('R4  Heartbleed/Wednesday', df_R4),
    ('R5  WebAttack/Thursday',   df_R5),
    ('R6  Infiltration/Thursday',df_R6),
    ('R7  Botnet/Friday',        df_R7),
    ('R8  DDoS/Friday',          df_R8),
    ('R9  PortScan/Friday',      df_R9),
]

for nombre, part in ataque_parts:
    n = conteos[nombre]
    print(f'\n' + '='*70)
    print(f'  PARTICIÓN {nombre}')
    print(f'  Registros: {n:,} | P = {n/total:.4f}')
    print('='*70)

    print('\n Distribución de etiquetas (Label) dentro de la partición')
    part.groupBy('Label').count() \
        .orderBy('count', ascending=False).show(10, truncate=False)

    print('Muestra de 5 registros')
    part.select(COLS_DEMO).show(5, truncate=True)

    print('Estadísticas de Flow Duration y Flow Packets/s')
    part.select('Flow Duration', 'Flow Packets/s') \
        .describe().show(truncate=False)


  PARTICIÓN R2  BruteForce/Tuesday
  Registros: 13,835 | P = 0.0049

 Distribución de etiquetas (Label) dentro de la partición
+-----------+-----+
|Label      |count|
+-----------+-----+
|FTP-Patator|7938 |
|SSH-Patator|5897 |
+-----------+-----+

Muestra de 5 registros
+-----------+------------+--------+-------------+-----------------+----------------------+--------------+
|      Label|attack_group|day_name|Flow Duration|Total Fwd Packets|Total Backward Packets|Flow Packets/s|
+-----------+------------+--------+-------------+-----------------+----------------------+--------------+
|FTP-Patator|  BruteForce| Tuesday|      5216127|                3|                     1|   0.766852494|
|FTP-Patator|  BruteForce| Tuesday|           20|                1|                     1|      100000.0|
|FTP-Patator|  BruteForce| Tuesday|           38|                1|                     1|   52631.57895|
|FTP-Patator|  BruteForce| Tuesday|           80|                1|                     1|  

---
## Sección 16: Demostración de Filtros Adicionales por Partición

Para ilustrar la flexibilidad del esquema de particionamiento, aplicamos filtros adicionales sobre particiones específicas. Esto demuestra que cada partición puede ser consultada de forma independiente para análisis más granulares — por ejemplo, aislar solo los flujos de alta intensidad dentro de la partición DoS, o los flujos de corta duración dentro de BENIGN/Monday.

In [16]:
# Ejemplo 1: Flujos de muy alta intensidad dentro de R3 (DoS)
print('Ejemplo 1: Flujos DoS con Flow Packets/s > 1000 (ataques de alta intensidad)')
df_R3_alta = df_R3.filter(F.col('Flow Packets/s') > 1000)
print(f'Registros DoS con >1000 pkt/s: {df_R3_alta.count():,}')
df_R3_alta.groupBy('Label').count() \
    .orderBy('count', ascending=False).show(truncate=False)
df_R3_alta.select(COLS_DEMO).show(5, truncate=True)

Ejemplo 1: Flujos DoS con Flow Packets/s > 1000 (ataques de alta intensidad)
Registros DoS con >1000 pkt/s: 67,744
+----------------+-----+
|Label           |count|
+----------------+-----+
|DoS Hulk        |66744|
|DoS slowloris   |624  |
|DoS Slowhttptest|352  |
|DoS GoldenEye   |24   |
+----------------+-----+

+-------------+------------+---------+-------------+-----------------+----------------------+--------------+
|        Label|attack_group| day_name|Flow Duration|Total Fwd Packets|Total Backward Packets|Flow Packets/s|
+-------------+------------+---------+-------------+-----------------+----------------------+--------------+
|DoS slowloris|         DoS|Wednesday|          229|                2|                     0|   8733.624454|
|DoS slowloris|         DoS|Wednesday|          214|                2|                     0|   9345.794393|
|DoS slowloris|         DoS|Wednesday|          799|                1|                     2|   3754.693367|
|DoS slowloris|         DoS|We

In [17]:
# Ejemplo 2: Flujos de corta duración dentro de R9 (PortScan)
print('Ejemplo 2: Flujos PortScan con duración < 1 segundo (escaneos rápidos)')
# Flow Duration está en microsegundos: 1 segundo = 1,000,000 µs
df_R9_corto = df_R9.filter(F.col('Flow Duration') < 1_000_000)
print(f'Registros PortScan con duración < 1s: {df_R9_corto.count():,}')
df_R9_corto.select(COLS_DEMO).show(5, truncate=True)

Ejemplo 2: Flujos PortScan con duración < 1 segundo (escaneos rápidos)
Registros PortScan con duración < 1s: 158,600
+--------+------------+--------+-------------+-----------------+----------------------+--------------+
|   Label|attack_group|day_name|Flow Duration|Total Fwd Packets|Total Backward Packets|Flow Packets/s|
+--------+------------+--------+-------------+-----------------+----------------------+--------------+
|PortScan|    PortScan|  Friday|           70|                1|                     1|   28571.42857|
|PortScan|    PortScan|  Friday|           52|                1|                     1|   38461.53846|
|PortScan|    PortScan|  Friday|          101|                1|                     1|    19801.9802|
|PortScan|    PortScan|  Friday|          671|                2|                     1|   4470.938897|
|PortScan|    PortScan|  Friday|          102|                1|                     1|   19607.84314|
+--------+------------+--------+-------------+-------------

In [18]:
# Ejemplo 3: Comparación estadística entre partición benigna y de ataque del mismo día
print('Ejemplo 3: Comparación BENIGN vs DoS del miércoles')
print('\nEstadísticas BENIGN/Wednesday (R11):')
df_R11.select('Flow Duration', 'Total Fwd Packets', 'Flow Packets/s') \
      .describe().show(truncate=False)

print('Estadísticas DoS/Wednesday (R3):')
df_R3.select('Flow Duration', 'Total Fwd Packets', 'Flow Packets/s') \
     .describe().show(truncate=False)

Ejemplo 3: Comparación BENIGN vs DoS del miércoles

Estadísticas BENIGN/Wednesday (R11):
+-------+--------------------+------------------+------------------+
|summary|Flow Duration       |Total Fwd Packets |Flow Packets/s    |
+-------+--------------------+------------------+------------------+
|count  |440010              |440010            |439662            |
|mean   |1.2095962512381537E7|11.913695143292198|60911.68491435843 |
|stddev |3.1325059594998468E7|937.4102322332737 |232483.96585098625|
|min    |0                   |1                 |0.016721809       |
|max    |119999998           |203943            |3000000.0         |
+-------+--------------------+------------------+------------------+

Estadísticas DoS/Wednesday (R3):
+-------+-------------------+------------------+------------------+
|summary|Flow Duration      |Total Fwd Packets |Flow Packets/s    |
+-------+-------------------+------------------+------------------+
|count  |252661             |252661            |2517

---
## Sección 17: Extracción de Sub-muestras de Verificación

Para verificar el funcionamiento del código de particionamiento, extraemos sub-muestras de prueba (~1,000 registros por partición) usando **Muestreo Aleatorio Simple sin reemplazo** con `seed=42` para garantizar reproducibilidad. Para las particiones ultra-minoritarias R4 (Heartbleed, n=11) y R6 (Infiltration, n=36), la fracción es 1.0 → **muestreo censal completo**.

> **Nota:** Estas sub-muestras son solo de verificación. Los conjuntos definitivos de entrenamiento y prueba se generarán en la siguiente etapa con fracciones calculadas estadísticamente.

In [19]:
SEED = 42
TARGET = 1000

print(f'{"Partición":<32} {"N partición":>12} {"Fracción":>10} {"N muestra":>10}')
print('-' * 66)

muestras = {}
for nombre, part in particiones.items():
    n = conteos[nombre]
    frac = min(TARGET / n, 1.0) if n > 0 else 0
    muestra = part.sample(withReplacement=False, fraction=frac, seed=SEED)
    n_muestra = muestra.count()
    muestras[nombre] = muestra
    censo = ' ← CENSO COMPLETO' if frac == 1.0 else ''
    print(f'{nombre:<32} {n:>12,} {frac:>10.4f} {n_muestra:>10,}{censo}')

Partición                         N partición   Fracción  N muestra
------------------------------------------------------------------
R1  BENIGN/Monday                     529,903     0.0019      1,019
R2  BruteForce/Tuesday                 13,835     0.0723      1,016
R3  DoS/Wednesday                     252,661     0.0040      1,002
R4  Heartbleed/Wednesday                   11     1.0000         11 ← CENSO COMPLETO
R5  WebAttack/Thursday                  2,180     0.4587      1,021
R6  Infiltration/Thursday                  36     1.0000         36 ← CENSO COMPLETO
R7  Botnet/Friday                       1,966     0.5086      1,003
R8  DDoS/Friday                       128,027     0.0078      1,016
R9  PortScan/Friday                   158,930     0.0063      1,055
R10 BENIGN/Tuesday                    432,057     0.0023        965
R11 BENIGN/Wednesday                  440,010     0.0023      1,003
R12 BENIGN/Thursday                   456,737     0.0022      1,020
R13 BENIGN/Frid

---
## Sección 18: Verificación de Distribución de Etiquetas en Sub-muestras de Ataque

Validación final: confirmamos que las sub-muestras de las particiones de ataque conservan la diversidad de sub-clases de `Label`. En particular verificamos que R4 (Heartbleed) y R6 (Infiltration) incluyen todos sus registros (censo completo), y que R5 (WebAttack) conserva las tres sub-clases incluyendo SQL Injection (ultra-minoritaria dentro de la partición).

In [20]:
ataques_verificar = [
    'R2  BruteForce/Tuesday',
    'R3  DoS/Wednesday',
    'R4  Heartbleed/Wednesday',
    'R5  WebAttack/Thursday',
    'R6  Infiltration/Thursday',
    'R7  Botnet/Friday',
    'R8  DDoS/Friday',
    'R9  PortScan/Friday',
]

for nombre in ataques_verificar:
    n_original = conteos[nombre]
    n_muestra  = muestras[nombre].count()
    print(f'\n Sub-muestra {nombre}')
    print(f'    Original: {n_original:,} registros | Muestra: {n_muestra:,} registros')
    muestras[nombre].groupBy('Label') \
                    .count() \
                    .orderBy('count', ascending=False) \
                    .show(10, truncate=False)


 Sub-muestra R2  BruteForce/Tuesday
    Original: 13,835 registros | Muestra: 1,016 registros
+-----------+-----+
|Label      |count|
+-----------+-----+
|FTP-Patator|588  |
|SSH-Patator|428  |
+-----------+-----+


 Sub-muestra R3  DoS/Wednesday
    Original: 252,661 registros | Muestra: 1,002 registros
+----------------+-----+
|Label           |count|
+----------------+-----+
|DoS Hulk        |931  |
|DoS GoldenEye   |38   |
|DoS Slowhttptest|19   |
|DoS slowloris   |14   |
+----------------+-----+


 Sub-muestra R4  Heartbleed/Wednesday
    Original: 11 registros | Muestra: 11 registros
+----------+-----+
|Label     |count|
+----------+-----+
|Heartbleed|11   |
+----------+-----+


 Sub-muestra R5  WebAttack/Thursday
    Original: 2,180 registros | Muestra: 1,021 registros
+-------------------------+-----+
|Label                    |count|
+-------------------------+-----+
|Web Attack  Brute Force  |703  |
|Web Attack  XSS          |310  |
|Web Attack  Sql Injection|8    |
+-------

---
## Sección 19: Resumen Final del Particionamiento

Tabla resumen con todos los resultados obtenidos: número de registros por partición, probabilidad empírica, tamaño de la sub-muestra de verificación y técnica de muestreo aplicada.

In [21]:
print('=' * 90)
print('RESUMEN FINAL DEL PARTICIONAMIENTO — CIC-IDS2017')
print('=' * 90)
print(f'{"Regla":<8} {"Descripción":<28} {"N registros":>12} {"P empírica":>11} '
      f'{"N muestra":>10} {"Técnica":>18}')
print('-' * 90)

tecnicas = {
    'R1':  'SRS sin reemplazo',
    'R2':  'Stratified por Label',
    'R3':  'Stratified por Label',
    'R4':  'CENSO (n=11)',
    'R5':  'Stratified por Label',
    'R6':  'CENSO (n=36)',
    'R7':  'Stratified por Label',
    'R8':  'Stratified por Label',
    'R9':  'Stratified por Label',
    'R10': 'SRS sin reemplazo',
    'R11': 'SRS sin reemplazo',
    'R12': 'SRS sin reemplazo',
    'R13': 'SRS sin reemplazo',
}

desc = {
    'R1':  'BENIGN/Monday',
    'R2':  'BruteForce/Tuesday',
    'R3':  'DoS/Wednesday',
    'R4':  'Heartbleed/Wednesday',
    'R5':  'WebAttack/Thursday',
    'R6':  'Infiltration/Thursday',
    'R7':  'Botnet/Friday',
    'R8':  'DDoS/Friday',
    'R9':  'PortScan/Friday',
    'R10': 'BENIGN/Tuesday',
    'R11': 'BENIGN/Wednesday',
    'R12': 'BENIGN/Thursday',
    'R13': 'BENIGN/Friday',
}

suma_total = 0
for nombre, part in particiones.items():
    regla = nombre.split()[0].strip()
    n = conteos[nombre]
    n_m = muestras[nombre].count()
    suma_total += n
    tec = tecnicas.get(regla, 'SRS')
    des = desc.get(regla, '')
    print(f'{regla:<8} {des:<28} {n:>12,} {n/total:>11.4f} {n_m:>10,} {tec:>18}')

print('-' * 90)
print(f'{"TOTAL":<8} {"13 particiones":<28} {suma_total:>12,} {suma_total/total:>11.4f}')
print('=' * 90)
print(f'\n Suma de probabilidades = {suma_total/total:.4f}')
print(f' Dataset post-limpieza:   {total:,} registros')
print(f' Seed de reproducibilidad: {SEED}')
print('\nParticionamiento MUTUAMENTE EXCLUYENTE y COLECTIVAMENTE EXHAUSTIVO confirmado.')

RESUMEN FINAL DEL PARTICIONAMIENTO — CIC-IDS2017
Regla    Descripción                   N registros  P empírica  N muestra            Técnica
------------------------------------------------------------------------------------------
R1       BENIGN/Monday                     529,903      0.1872      1,019  SRS sin reemplazo
R2       BruteForce/Tuesday                 13,835      0.0049      1,016 Stratified por Label
R3       DoS/Wednesday                     252,661      0.0893      1,002 Stratified por Label
R4       Heartbleed/Wednesday                   11      0.0000         11       CENSO (n=11)
R5       WebAttack/Thursday                  2,180      0.0008      1,021 Stratified por Label
R6       Infiltration/Thursday                  36      0.0000         36       CENSO (n=36)
R7       Botnet/Friday                       1,966      0.0007      1,003 Stratified por Label
R8       DDoS/Friday                       128,027      0.0452      1,016 Stratified por Label
R9       Port

---
## Cierre de la Sesión de Spark

In [22]:
spark.stop()
print('Sesión de Spark cerrada correctamente.')

Sesión de Spark cerrada correctamente.
